# DARKEN Reanalysis with CalciMetrics 2.0

Reanalysis of 4 acute brain-slice recordings (experiments 725, 732, 734, 736) from the DARKEN sDARKEN characterisation dataset.

**Pipeline** (identical to the original `all/imaging_analysis_darken_slice_all.ipynb`):  
`raw CSV` → background subtraction → ΔF/F (median baseline) → exclude bad ROIs → combine all ROIs → linear detrend → Gaussian smooth (window=5, σ=2.5) → % ΔF/F

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from calcimetrics import Dataset, load_imagej_csv
from calcimetrics.core import Recording
from calcimetrics.normalization import compute_dff, to_percent
from calcimetrics.detrending import detrend_linear
from calcimetrics.smoothing import gaussian_smooth
from calcimetrics.plotting import plot_overlay, plot_mean_with_error, plot_heatmap

In [ ]:
# ── Plot style ───────────────────────────────────────────────────────────
# Set USE_NATURE_STYLE = True for a publication-ready Nature-journal look.
# Requires SciencePlots:  pip install SciencePlots
# 'no-latex' means LaTeX does not need to be installed locally.
# Set to False to use the default matplotlib style.

USE_NATURE_STYLE = True

import matplotlib as mpl

def apply_style():
    """Apply (or revert) the chosen style. Call again after seaborn imports."""
    if USE_NATURE_STYLE:
        try:
            import scienceplots  # noqa: F401
            plt.style.use(["science", "nature", "no-latex"])
            # A few extra tweaks for larger, more legible figures in a notebook
            mpl.rcParams.update({
                "figure.dpi":      150,
                "font.size":       9,
                "axes.labelsize":  10,
                "axes.titlesize":  10,
                "xtick.labelsize": 8,
                "ytick.labelsize": 8,
                "legend.fontsize": 8,
                "lines.linewidth": 1.0,
            })
        except ImportError:
            print("SciencePlots not found — using default style.\n"
                  "Install with:  pip install SciencePlots")
    else:
        plt.style.use("default")

apply_style()
print(f"Style: {'science + nature + no-latex' if USE_NATURE_STYLE else 'default'}  "
      f"| dpi = {mpl.rcParams['figure.dpi']:.0f}")
# ── Figure export ────────────────────────────────────────────────────────
# Set SAVE_FIGURES = True to export every plot as PNG + SVG.
SAVE_FIGURES = True
FIGURES_DIR  = Path.cwd().parent / "figures" / "darken"

if SAVE_FIGURES:
    FIGURES_DIR.mkdir(exist_ok=True)
    print(f"Figures will be saved to: {FIGURES_DIR}")

def save_fig(fig, name: str) -> None:
    """Save *fig* as <name>.png and <name>.svg inside FIGURES_DIR."""
    if not SAVE_FIGURES:
        return
    for ext in ("png", "svg"):
        fig.savefig(FIGURES_DIR / f"{name}.{ext}",
                    dpi=300, bbox_inches="tight")
    print(f"  saved  {name}.png / .svg")


## 1. Load recordings

Each entry specifies the raw CSV path and the ROI columns to exclude by name (matching the `_cleaned.csv` files in `all/cleaned/`).

In [ ]:
ROOT = Path.cwd().parent
data_root = ROOT / "input_data" / "darken"

# Exclude lists: ROI column names removed in the original analysis.
configs = {
    "725": {
        "file": "Results_725.csv",
        "exclude": ["Mean6", "Mean7", "Mean8", "Mean9", "Mean11"],  # keep Mean1-5, Mean10
        "bio_rep": "slice_A",
    },
    "732": {
        "file": "Results_Experiment_732.csv",
        "exclude": ["Mean3", "Mean5", "Mean6"],  # keep Mean1,2,4,7-11
        "bio_rep": "slice_B",
    },
    "734": {
        "file": "Results_Experiment_734.csv",
        "exclude": ["Mean5", "Mean6", "Mean9", "Mean12"],  # keep Mean1-4,7,8,10,11,13-15
        "bio_rep": "slice_C",
    },
    "736": {
        "file": "Results_736.csv",
        "exclude": ["Mean4"],  # keep Mean1-3, Mean5-11
        "bio_rep": "slice_D",
    },
}

In [ ]:
slices = Dataset()

for rec_id, cfg in configs.items():
    path = data_root / rec_id / cfg["file"]
    rec = load_imagej_csv(
        path, fps=0.5, name=rec_id,
        biological_replicate=cfg["bio_rep"],
        condition="sDARKEN_slice",
    )
    rec.traces = compute_dff(rec.traces, method="median")
    if cfg["exclude"]:
        rec = rec.exclude_rois(cfg["exclude"])
    slices.add(rec)
    print(f"  {rec_id}: {rec.traces.shape[1]} ROIs — {list(rec.traces.columns)}")

print()
print(slices.summary())

## 2. Combine, detrend, smooth

All ROIs from all 4 recordings are pooled into one DataFrame (`level="roi"` = 35 cells).

Processing matches the original pipeline:
1. Linear detrend (`scipy.signal.detrend`)
2. Gaussian smooth (window = 5 frames, σ = 2.5)
3. Convert to % ΔF/F

A temporary `Recording` is created from the processed DataFrame so all `calcimetrics.plotting` functions can be used directly.

In [ ]:
# Pool all 35 ROIs (601 frames × 35 columns).
all_rois = slices.combine_traces(level="roi")

# Pipeline: linear detrend → Gaussian smooth → % ΔF/F
all_rois = detrend_linear(all_rois)
all_rois = gaussian_smooth(all_rois, window_size=5, sigma=2.5)
all_rois = to_percent(all_rois)

n_rois = all_rois.shape[1]

# Wrap the processed DataFrame in a Recording so the plotting helpers work.
all_rec = Recording(
    name=f"all slices (n = {n_rois} ROIs)",
    raw=all_rois,
    traces=all_rois,
    fps=0.5,
    excluded_rois=[],
    biological_replicate="pooled",
    condition="sDARKEN_slice",
)

print(f"Shape: {all_rois.shape}  "
      f"Time: 0–{all_rec.duration_s:.0f} s  "
      f"Value range: {all_rois.values.min():.1f} – {all_rois.values.max():.1f} % ΔF/F")

## 3. Plots

Three figures that reproduce the outputs saved in `all/cleaned/`,  
now using the `calcimetrics.plotting` functions.

In [ ]:
# Plot 1: All individual ROIs overlaid — slices_all_individual_rois
fig = plot_overlay(all_rec, ylabel="% ΔF/F", alpha=0.5)
save_fig(fig, "01_all_rois_overlay")
plt.show()


In [ ]:
# Plot 2: Mean ± SD — slices_all_mean_SD
fig = plot_mean_with_error(all_rec, error="sd", ylabel="% ΔF/F")
save_fig(fig, "02_all_rois_mean_sd")
plt.show()


In [ ]:
# Plot 3: Heatmap ROI × time — slices_all_heatmap
fig = plot_heatmap(all_rec, cbar_label="% ΔF/F", figsize=(10, 8))
save_fig(fig, "03_all_rois_heatmap")
plt.show()


## 4. Per-slice plots

The grand mean across all 4 slices in Section 3 is not very meaningful for spontaneous activity — each slice fires independently, so individual events average out.  
The plots below apply the same pipeline (detrend → smooth → % ΔF/F) to each recording separately and show all-ROI overlay and mean ± SD per slice.

In [ ]:
for rec_id, rec in slices.recordings.items():
    # Apply the same processing as for the pooled data.
    traces = detrend_linear(rec.traces.copy())
    traces = gaussian_smooth(traces, window_size=5, sigma=2.5)
    traces = to_percent(traces)

    proc_rec = Recording(
        name=f"Slice {rec_id}  ({traces.shape[1]} ROIs)",
        raw=traces,
        traces=traces,
        fps=rec.fps,
        excluded_rois=rec.excluded_rois,
        biological_replicate=rec.biological_replicate,
        condition=rec.condition,
    )

    fig = plot_overlay(proc_rec, ylabel="% ΔF/F", alpha=0.6)
    save_fig(fig, f"04_slice_{rec_id}_overlay")
    plt.show()

    fig = plot_mean_with_error(proc_rec, error="sd", ylabel="% ΔF/F")
    save_fig(fig, f"05_slice_{rec_id}_mean_sd")
    plt.show()


## 5. Event analysis

sDARKEN fluorescence **decreases** (darkens) when serotonin binds, so spontaneous serotonin release events appear as **downward deflections** (negative % ΔF/F).

The setup cell below detects those events for every ROI using a −2 SD threshold and a minimum inter-event spacing of 20 s. The five plot cells that follow each tell a different part of the story.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.signal import find_peaks
from scipy.stats import expon

FPS = 0.5
SD_THRESHOLD = 2.0   # events must dip ≥ 2 SD below the ROI mean
MIN_DIST_S   = 20    # minimum 20 s between consecutive events
MIN_DIST_F   = int(MIN_DIST_S * FPS)

# Re-build per-slice processed recordings and detect events.
proc_recs = {}
events    = {}   # {rec_id: {col: array_of_frame_indices}}

for rec_id, rec in slices.recordings.items():
    tr = detrend_linear(rec.traces.copy())
    tr = gaussian_smooth(tr, window_size=5, sigma=2.5)
    tr = to_percent(tr)
    proc_recs[rec_id] = Recording(
        name=f"Slice {rec_id}",
        raw=tr, traces=tr, fps=rec.fps,
        excluded_rois=rec.excluded_rois,
        biological_replicate=rec.biological_replicate,
        condition=rec.condition,
    )
    events[rec_id] = {}
    for col in tr.columns:
        s = tr[col]
        # height in negated signal that corresponds to ≥ SD_THRESHOLD*std below mean
        min_h = -s.mean() + SD_THRESHOLD * s.std()
        peaks, _ = find_peaks(-s.values, height=min_h, distance=MIN_DIST_F)
        events[rec_id][col] = peaks

total = sum(len(v) for d in events.values() for v in d.values())
print(f"Detected {total} events  (threshold: −{SD_THRESHOLD} SD, min spacing: {MIN_DIST_S} s)")
for rid in events:
    n = sum(len(v) for v in events[rid].values())
    print(f"  Slice {rid}: {n} events across {len(events[rid])} ROIs")

# Re-apply style after seaborn (seaborn can reset rcParams)
apply_style()


In [ ]:
# ── Plot 5a: Event raster plot ───────────────────────────────────────────
colors    = plt.cm.Set2.colors
slice_ids = list(events.keys())
cmap_     = {rid: colors[i % len(colors)] for i, rid in enumerate(slice_ids)}

fig, ax = plt.subplots(figsize=(12, 6))

y_pos   = 0
yticks  = []
ylabels = []
slice_y_centers = {}

for rid in slice_ids:
    y_start = y_pos
    for col, frames in events[rid].items():
        t_events = frames / FPS
        ax.scatter(t_events, [y_pos] * len(t_events),
                   marker='|', s=120, linewidth=1.8, color=cmap_[rid])
        yticks.append(y_pos)
        ylabels.append(col)
        y_pos += 1
    slice_y_centers[rid] = (y_start + y_pos - 1) / 2
    ax.axhspan(y_start - 0.5, y_pos - 0.5,
               color=cmap_[rid], alpha=0.08, zorder=0)
    y_pos += 1   # gap between slices

# Slice labels on right y-axis
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())
ax2.set_yticks(list(slice_y_centers.values()))
ax2.set_yticklabels([f"Slice {rid}" for rid in slice_y_centers], fontsize=9)
ax2.spines['right'].set_visible(False)

ax.set_yticks(yticks)
ax.set_yticklabels(ylabels, fontsize=7)
ax.set_xlabel("Time (s)")
ax.set_title("Spontaneous serotonin release events — raster plot")
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.tight_layout()
save_fig(fig, "06_event_raster")
plt.show()


In [ ]:
# ── Plot 5b: Average event waveform (aligned to peak) ───────────────────
WINDOW_S = 30          # ± 30 s window around each event peak
WINDOW_F = int(WINDOW_S * FPS)   # in frames

all_waves = []
per_slice = {rid: [] for rid in events}

for rid in events:
    tr = proc_recs[rid].traces
    for col, frames in events[rid].items():
        s = tr[col].values
        for pf in frames:
            start, end = pf - WINDOW_F, pf + WINDOW_F + 1
            if start >= 0 and end <= len(s):
                all_waves.append(s[start:end])
                per_slice[rid].append(s[start:end])

if not all_waves:
    print("No complete waveforms found — try relaxing the threshold.")
else:
    t_wave = np.arange(-WINDOW_F, WINDOW_F + 1) / FPS
    mat    = np.array(all_waves)

    fig, ax = plt.subplots(figsize=(8, 5))

    # Individual waveforms in light grey
    for w in mat:
        ax.plot(t_wave, w, color='grey', alpha=0.12, lw=0.6)

    # Per-slice means
    colors_ = plt.cm.Set2.colors
    for i, (rid, waves) in enumerate(per_slice.items()):
        if waves:
            m = np.array(waves).mean(axis=0)
            ax.plot(t_wave, m, color=colors_[i], lw=1.5, alpha=0.8,
                    label=f"Slice {rid} (n={len(waves)})")

    # Grand mean ± SEM
    grand_mean = mat.mean(axis=0)
    grand_sem  = mat.std(axis=0) / np.sqrt(len(mat))
    ax.plot(t_wave, grand_mean, color='black', lw=2.5,
            label=f"Grand mean  (n = {len(mat)} events)")
    ax.fill_between(t_wave, grand_mean - grand_sem, grand_mean + grand_sem,
                    color='black', alpha=0.25)

    ax.axhline(0, color='k', ls='--', lw=0.6)
    ax.axvline(0, color='red', ls='--', lw=1.0, label='Event peak')
    ax.set_xlabel("Time relative to event peak (s)")
    ax.set_ylabel("% ΔF/F")
    ax.set_title("Average sDARKEN event waveform")
    ax.legend(frameon=False, fontsize=8)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    plt.tight_layout()
    save_fig(fig, "07_event_waveform")
    plt.show()
